In [24]:
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
from datetime import datetime, timedelta


In [20]:
def extract_his_2d(vars,ilats,ilons,fn_his):
    data_out = dict()
    with nc.Dataset(fn_his) as ds:
        for var in vars:
            data = ds.variables[var][:,ilats,ilons]
            data_out[var] = data
        data_out['ocean_time'] = ds.variables['ocean_time'][:]

    return data_out

def dlat_to_dy(dlat,alat):
#
# dy   = latitude difference in meters
# dlat = latidute difference in degrees
# alat = average latitude between the two fixes
# Reference: American Practical Navigator, Vol II, 1975 Edition, p 5

    rlat = alat * np.pi/180
    m = (111132.09 * rlat - 
        566.05 * np.cos(2 * rlat) + 1.2 * np.cos(4 * rlat) )
    dy = m * dlat
    return dy

def dlon_to_dx(dlon,alat):
#
# dlon = longitude difference in degrees
# dx   = longitude difference in meters
# alat = average latitude between the two fixes

    rlat = alat * np.pi/180
    p = 111415.13 * np.cos(rlat) - 94.55 * np.cos(3 * rlat)
    dx = p * dlon 
    return dx


def get_lat_lon_indices(lats,lons,lvl):
    if lvl == 'LV1':
        fn_grd = '/dataSIO/PFM_Simulations/Grid/GRID_SDTJRE_LV1_rx020_hmask.nc'
    elif lvl == 'LV2':
        fn_grd = '/dataSIO/PFM_Simulations/Grid/GRID_SDTJRE_LV2_rx020.nc'
    elif lvl == 'LV3':
        fn_grd = '/dataSIO/PFM_Simulations/Grid/GRID_SDTJRE_LV3_rx020.nc'
    elif lvl == 'LV4':
        fn_grd = '/dataSIO/PFM_Simulations/Grid/GRID_SDTJRE_LV4_mss_oct2024.nc'

    with nc.Dataset(fn_grd) as ds:
        latg = ds.variables['lat_rho'][:]
        long = ds.variables['lon_rho'][:]
        mask_rho = ds.variables['mask_rho'][:]

    bar_lat = np.mean(latg)
    bar_lon = np.mean(long)
    xg = dlon_to_dx(long-bar_lon,bar_lat)
    yg = dlat_to_dy(latg-bar_lat,bar_lat)

    x0 = dlon_to_dx(lons-bar_lon,bar_lat)
    y0 = dlat_to_dy(lats-bar_lat,bar_lat)    

    cnt = 0
    ilt = []
    iln = []
    for y in y0:
        x = x0[cnt]
        dy = yg-y
        dx = xg-x
        D2 = np.square(dx) + np.square(dy)
        # Find the flattened index of the minimum value
        min_flat_index = np.argmin(D2)

        # Convert the flattened index to (row, column) indices
        row_index, col_index = np.unravel_index(min_flat_index, D2.shape)
        
        # check and see if mask_rho is 1 (ocean) here...
        land = True
        while land:
            msk = mask_rho[row_index,col_index]
            if msk == 1:
                land = False
                print('the (lat,lon) requested was: ',lats[cnt],',',lons[cnt])
                print('the (lat,lon) found is: ',latg[row_index,col_index],',',long[row_index,col_index])
            else:
                # move offshore and south!
                print('the lat,lon requested is land. moving offshore and south by 1')
                row_index = row_index - 1
                col_index = col_index - 1

        ilt.append(row_index)
        iln.append(col_index)

        cnt += 1    


    return ilt,iln

In [7]:

lats = [32.50,32.55]
lons = [-117.2,-117.2]
var = 'zeta'
t1 = '20241011' # start time of first file
t2 = '20241013' # start time of last file



In [ ]:
ilt3,iln3 = get_lat_lon_indices(lats,lons,'LV3')
print(ilt3,iln3)


the (lat,lon) requested was:  32.5 , -117.2
the (lat,lon) found is:  32.49941668873491 , -117.20084239467755
the (lat,lon) requested was:  32.55 , -117.2
the (lat,lon) found is:  32.550168363402186 , -117.20043268156094
[np.int64(121), np.int64(146)] [np.int64(165), np.int64(176)]


In [26]:

fn_his = '/scratch/PHM_Simulations/LV3_Forecast/His/LV3_ocean_his_202410110000.nc'
data = extract_his_2d(['zeta'],ilt3,iln3,fn_his)
t = datetime(1999,1,1) + data['ocean_time'][:]*timedelta(days=1)
zeta = data['zeta'][:]

fig, ax = plt.subplots()
ax.plot(t,zeta)



OverflowError: date value out of range

In [18]:
dum = np.random.rand(7,6)
dum2 = dum[:,[2,4,5]]
print(dum2)

[[0.72904033 0.6452015  0.47984805]
 [0.52931798 0.64424046 0.50368832]
 [0.19018229 0.03266774 0.24260252]
 [0.40447052 0.79987138 0.37553142]
 [0.54940102 0.32158055 0.28291318]
 [0.43405287 0.89852642 0.65944622]
 [0.99303061 0.0601309  0.65071681]]
